# Topic: SQL: Monthly Active Users (MAU) & MoM Growth/Retention

## Definition (30-second explanation)
* MAU measures the total number of unique users who interacted with a product or performed a specific action within a given calendar month.
* It is a foundational product analytics KPI used to track overarching product growth, evaluate feature success, and model revenue.
* It is calculated by grouping data by a truncated month and taking a distinct count of user identifiers.

## Why Interviewers Ask This
* **Technical Signal:** Tests fundamental SQL aggregations, date manipulation (`DATE_TRUNC`, `EXTRACT`), and deduplication (`DISTINCT`).
* **Advanced SQL:** Often used as a stepping stone to test window functions (e.g., using `LAG()` for Month-over-Month growth).
* **Product Sense:** Assesses if you understand the business context—knowing to ask how "active" is defined (e.g., simple login vs. making a purchase).

## Core Concepts
* **Date Truncation:** Standardizing granular timestamps into monthly buckets (e.g., mapping `2024-03-15 14:02:00` to `2024-03-01`).
* **Deduplication:** Using `COUNT(DISTINCT user_id)` ensures a user with 50 events in a month is only counted once, measuring *reach*, not *volume*.
* **Windowing for MoM:** Leveraging `LAG()` over a time-ordered window to compare current aggregations against previous periods.

## When to Use
* Tracking overall product growth and long-term user engagement trends.
* Calculating derived metrics like Month-over-Month (MoM) user growth or trial-to-paid MAU ratios.
* Building executive dashboards where daily fluctuations (DAU) are too noisy.

## Limitations
* Fails to capture daily volatility or granular session behavior (use DAU or Sessionization for this).
* Can present a false sense of growth if churn is high but offset by massive (and expensive) new user acquisition.
* Definitionally ambiguous: MAU numbers are meaningless if the "active" event is too passive (e.g., an automated push notification).

## Common Comparisons
* **MAU vs. DAU:** MAU shows broad reach; DAU shows daily habit. The DAU/MAU ratio (Stickiness) measures how often monthly users engage daily.
* **DATE_TRUNC vs. EXTRACT:** `DATE_TRUNC` keeps the year/month structure intact (`2024-01-01`), whereas `EXTRACT(MONTH)` isolates the month number (`1`), which will incorrectly group January 2024 and January 2025 together if not also grouped by year.

## Common Interview Traps
* **Missing DISTINCT:** Writing `COUNT(user_id)` instead of `COUNT(DISTINCT user_id)`, which double-counts repeat visitors.
* **Forgetting the Year:** Using `EXTRACT(MONTH)` without `EXTRACT(YEAR)` on multi-year datasets.
* **Blind Grouping:** Using `GROUP BY 1` without ensuring the date column is actually the first selected column, causing silent grouping errors.
* **Timezone Ignorance:** Failing to clarify how the system handles UTC vs. local timezones at the boundary of a month.

## SQL Syntax
```sql
-- Core MAU Pattern
SELECT 
    DATE_TRUNC('month', event_date) AS activity_month,
    COUNT(DISTINCT user_id) AS monthly_active_users
FROM user_events
GROUP BY 1
ORDER BY 1;

-- MoM Growth Pattern using CTE and LAG()
WITH monthly_mau AS (
    SELECT 
        DATE_TRUNC('month', event_date) AS activity_month,
        COUNT(DISTINCT user_id) AS mau
    FROM user_events
    GROUP BY 1
)
SELECT 
    activity_month,
    mau,
    LAG(mau) OVER (ORDER BY activity_month) AS prev_month_mau,
    ROUND(100.0 * (mau - LAG(mau) OVER (ORDER BY activity_month)) / 
          LAG(mau) OVER (ORDER BY activity_month), 2) AS mom_growth_pct
FROM monthly_mau;
```

## Important Formula
* **MoM Growth Rate:** `(Current Month MAU - Previous Month MAU) / Previous Month MAU`
* **Stickiness Ratio:** `DAU / MAU` (Values closer to 1 indicate a highly habitual product).

## 45-Second Interview Answer
"To calculate Monthly Active Users, I truncate the event timestamp to the month level using `DATE_TRUNC` and group by it. Inside the aggregation, I use `COUNT(DISTINCT user_id)` to guarantee each user is only counted once per month regardless of their activity volume. If asked for Month-over-Month growth, I wrap the MAU query in a CTE and apply the `LAG()` window function, ordering by month, to retrieve the previous month's MAU for the percentage calculation. Before writing the query, I would always verify with you how we are defining an 'active' event and what timezone boundaries we should respect."